# Exercises: Hypothesis Testing

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [18]:
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")
tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

- Hypothesis $H_1$: smoking is associated with time of visit

---

In [19]:
contingency_table = pd.crosstab(tips['smoker'], tips['time'])
print(contingency_table)

time    Lunch  Dinner
smoker               
Yes        23      70
No         45     106


In [20]:
chi2, p, dof, ex = chi2_contingency(contingency_table)
print(f"P-value: {p}")

P-value: 0.4771485672079724


In [21]:
if p < 0.05:
    # We reject the Null Hypothesis because the p-value is below the significance level (0.05)
    print("Reject the Null Hypothesis: There is a statistically significant association between smoking and time of visit.")
else:
    # We fail to reject the Null Hypothesis because there is not enough evidence to prove a relationship
    print("Fail to reject the Null Hypothesis: There is no statistically significant evidence of an association between smoking and time of visit.")

Fail to reject the Null Hypothesis: There is no statistically significant evidence of an association between smoking and time of visit.


- Hypothesis $H_2$: the bigger the group the higher the tip

In [22]:
from scipy.stats import pearsonr

# Calculate Pearson correlation
corr, p_value = pearsonr(tips['size'], tips['tip'])

print(f"Correlation Coefficient: {corr:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject the Null Hypothesis: There is a significant positive correlation between group size and tip amount.")
else:
    print("Fail to reject the Null Hypothesis: There is no significant correlation between group size and tip amount.")

Correlation Coefficient: 0.4893
P-value: 0.0000
Reject the Null Hypothesis: There is a significant positive correlation between group size and tip amount.


- Hypothesis $H_3$: group size is different based on the time of visit

In [23]:
from scipy.stats import ttest_ind

# Split the data into two groups
lunch_size = tips[tips['time'] == 'Lunch']['size']
dinner_size = tips[tips['time'] == 'Dinner']['size']

# Perform Independent T-test
t_stat, p_val = ttest_ind(lunch_size, dinner_size)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Reject the Null Hypothesis: Group size significantly differs based on the time of visit.")
else:
    print("Fail to reject the Null Hypothesis: There is no significant difference in group size between lunch and dinner.")

T-statistic: -1.6174
P-value: 0.1071
Fail to reject the Null Hypothesis: There is no significant difference in group size between lunch and dinner.


- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)

In [24]:
male_bill = tips[tips['sex'] == 'Male']['total_bill']
female_bill = tips[tips['sex'] == 'Female']['total_bill']
t_stat, p_val = stats.ttest_ind(male_bill, female_bill)

print(f"H4 (Gender vs Bill) P-value: {p_val:.4f}")
if p_val < 0.05:
    print("Reject H4 Null: Total bill amount differs by gender.")
else:
    print("Fail to reject H4 Null: No significant difference in bill amount between genders.")

H4 (Gender vs Bill) P-value: 0.0236
Reject H4 Null: Total bill amount differs by gender.


- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

In [25]:
# Create a new column for tip percentage
tips['tip_pct'] = tips['tip'] / tips['total_bill']

# Check correlation between size and tip percentage
corr_pct, p_val_pct = pearsonr(tips['size'], tips['tip_pct'])

print(f"Correlation (Size vs Tip %): {corr_pct:.4f}")
print(f"P-value: {p_val_pct:.4f}")

if p_val_pct > 0.05:
    print("Conclusion: Group size is likely a confounder. It increases the total bill, which in turn increases the tip, but it doesn't necessarily increase the tip percentage.")
else:
    print("Conclusion: Group size has a direct effect on the tip percentage itself.")

Correlation (Size vs Tip %): -0.1429
P-value: 0.0256
Conclusion: Group size has a direct effect on the tip percentage itself.
